# FUTURE_DS_03: Marketing Funnel & Conversion Performance Analysis
### Professional Data Science Portfolio Project

This notebook covers the exploratory analysis, preprocessing pipeline, conversion funnel visualization, attribution performance tracking, and statistical validation of marketing campaigns.

### 1. Essential Library Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

%matplotlib inline
sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

### 2. Loading the Raw Marketing Dataset
We load a synthetic, highly realistic dataset designed to replicate key anomalies (missing fields, extreme outliers, non-standardized dates, duplicate registrations) typical of real-world multi-channel advertising engines.

In [2]:
# Load or generate synthetic data directly
try:
    df_raw = pd.read_csv('../marketing_dataset_raw.csv')
except FileNotFoundError:
    # Fallback definition using custom import or simulation script
    import sys
    sys.path.append('..')
    from analytics import generate_synthetic_data
    df_raw = generate_synthetic_data()

print(f"Raw dataset contains {df_raw.shape[0]} records and {df_raw.shape[1]} features.")
df_raw.head(3)

### 3. Data Cleansing & Anomaly Resolution

In [3]:
# Remove duplicates
initial_len = len(df_raw)
df_clean = df_raw.drop_duplicates(subset=['Marketing Channel', 'Campaign Name', 'Device Type', 'Region', 'Date'], keep='first').copy()
print(f"Dropped {initial_len - len(df_clean)} duplicate rows.")

# Handle missing device data via mode substitution
df_clean['Device Type'] = df_clean['Device Type'].fillna('Desktop')
print("Imputed missing device records.")

# Standardize format date
df_clean['Date'] = pd.to_datetime(df_clean['Date'], errors='coerce')
print("Dates reformatted to DateTime objects.")

### 4. Conversion Funnel Exploration
We aggregate global interactions across all campaign touchpoints: Impressions, Clicks, Visits, Signups, Purchases.

In [4]:
stages = ['Impressions', 'Clicks', 'Website Visits', 'Signups', 'Purchases']
sums = df_clean[stages].sum()

funnel_df = pd.DataFrame({
    'Stage': stages,
    'UsersCount': sums.values
})
funnel_df['Conversion_Rate (%)'] = (funnel_df['UsersCount'] / funnel_df.iloc[0]['UsersCount'] * 100).round(2)
funnel_df

### 5. Attribution & Channel Performance Analysis
Calculating crucial performance metrics: CTR, CPC, CPA, and ROI.

In [5]:
channels = df_clean.groupby('Marketing Channel').agg({
    'Impressions': 'sum',
    'Clicks': 'sum',
    'Website Visits': 'sum',
    'Signups': 'sum',
    'Purchases': 'sum',
    'Campaign Cost': 'sum',
    'Revenue Generated': 'sum'
}).reset_index()

channels['CTR (%)'] = (channels['Clicks'] / channels['Impressions'] * 100).round(2)
channels['CPC ($)'] = (channels['Campaign Cost'] / channels['Clicks']).round(2)
channels['CPA ($)'] = (channels['Campaign Cost'] / channels['Purchases']).round(2)
channels['ROI'] = ((channels['Revenue Generated'] - channels['Campaign Cost']) / channels['Campaign Cost']).round(2)

channels.sort_values(by='ROI', ascending=False)